In [ ]:
%pip install chromadb==1.3.5 ipykernel==7.1.0 langchain==0.3.27 langchain-community==0.3.31 langchain-core==0.3.80 langchain-huggingface==0.3.1 langchain-openai==0.3.35 langchain-text-splitters==0.3.11 pypdf==6.3.0 python-dotenv==1.2.1 rank-bm25==0.2.2 sentence-transformers==5.1.2

# Chat with Text (RAG)
This notebook implements a Retrieval-Augmented Generation (RAG) pipeline to answer questions about the State of the Union address using a local embedding model and Llama 3 via OpenRouter.

In [1]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

/home/miad/projects/RAG/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Setup Environment
Define API keys for OpenRouter.

In [2]:
# Get key from: https://openrouter.ai/keys
os.environ["OPENAI_API_KEY"] = "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" # Your OpenRouter Key
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

### 1. Load Documents
Read the text file into memory.

In [3]:
# 1. Load Data
loader = TextLoader("./state_of_the_union.txt")
docs = loader.load()

### 2. Split Text
Break the document into chunks for processing.

In [4]:
# 2. Split Data
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

### 3. Create the TWO Retrievers
Initialize both Semantic Search (Vector) and Keyword Search (BM25) retrievers.

In [5]:
# Retriever A: The Semantic Search (Vector)
print("Initializing Vector Search...")
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

Initializing Vector Search...


In [6]:
# Retriever B: The Keyword Search (BM25)
print("Initializing Keyword Search...")
keyword_retriever = BM25Retriever.from_documents(splits)
keyword_retriever.k = 2 

Initializing Keyword Search...


### 4. The "Ensemble"
Merge the two retrievers.

In [7]:
# 4. The "Ensemble" (The Manager that merges them)
print("Merging retrievers...")
ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, keyword_retriever],
    weights=[0.5, 0.5] 
)

Merging retrievers...


### 5. Setup LLM
Initialize the model.

In [ ]:
# 5. Setup LLM
llm = ChatOpenAI(
    # model="meta-llama/llama-3-8b-instruct",
    model="x-ai/grok-4.1-fast",
    temperature=0
)

### 6. The RAG Chain
Create the pipeline using the ensemble retriever.

In [9]:
# 6. The RAG Chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": ensemble_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

### 7. Ask
Run a query to test the hybrid search.

In [10]:
query1 = "What did the president say about COVID-19?" 
response1 = rag_chain.invoke(query1)
print(f"Asking: {query1}")
print(f"\n--- ANSWER --- {response1}")
print(response1)

Asking: What did the president say about COVID-19?

--- ANSWER --- The president did not mention COVID-19 in the given text.
The president did not mention COVID-19 in the given text.


In [11]:
query2 = "What did the president say about Product #71-A?" 
response2 = rag_chain.invoke(query2)
print(f"Asking: {query2}")
print("\n--- ANSWER ---")
print(response2)

Asking: What did the president say about Product #71-A?

--- ANSWER ---
The president did not mention Product #71-A at all in the given context.


In [12]:
query3 = "What did the president say about Product #71-C?" 
response3 = rag_chain.invoke(query3)
print(f"Asking: {query3}")
print("\n--- ANSWER ---")
print(response3)

Asking: What did the president say about Product #71-C?

--- ANSWER ---
The president mentioned Product #71-C as a "great pump".
